# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from utils import load_pickle

from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-stellar-class/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
label_encoder = load_pickle('../models/label_encoder.pkl')

# Loading Datasets

In [3]:
# cols_to_drop = ['lg_0', 'lg_1', 'lg_2', 'extra_0', 'extra_1', 'extra_2', 'rf_0', 'rf_1', 'rf_2', 'hist_0', 'hist_1', 'hist_2']

X_train = pd.read_parquet('../data/X_train_stacking_layer_two.parquet') #.drop(columns=cols_to_drop)
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_two.parquet') #.drop(columns=cols_to_drop)

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,lg_0,lg_1,lg_2,sgd_0,sgd_1,sgd_2
0,0.999910,0.000083,0.000007,0.999866,0.000124,0.000010,0.999928,0.000070,0.000003,0.988210,0.004367,0.007423,0.999210,0.000790,0.000000
1,0.992961,0.000411,0.006628,0.991422,0.000567,0.008011,0.992562,0.000566,0.006872,0.986847,0.004779,0.008373,0.988522,0.001245,0.010233
2,0.000109,0.999858,0.000033,0.000162,0.999819,0.000019,0.000030,0.999962,0.000008,0.009897,0.988895,0.001208,0.002604,0.997396,0.000000
3,0.999805,0.000187,0.000009,0.999766,0.000223,0.000010,0.999837,0.000159,0.000004,0.988202,0.004372,0.007426,0.999098,0.000902,0.000000
4,0.998393,0.001560,0.000047,0.998544,0.001413,0.000043,0.998238,0.001711,0.000051,0.987966,0.004405,0.007629,0.998207,0.001793,0.000000


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,lg_0,lg_1,lg_2,sgd_0,sgd_1,sgd_2
0,0.997841,0.002113,0.000045,0.997856,0.001945,0.000199,0.998030,0.001934,0.000036,0.987950,0.004507,0.007543,0.997783,0.002217,0.000000
1,0.996887,0.003080,0.000033,0.997599,0.002381,0.000020,0.996744,0.003240,0.000016,0.987886,0.004549,0.007565,0.997324,0.002676,0.000000
2,0.998244,0.001026,0.000731,0.998669,0.000563,0.000768,0.998349,0.000750,0.000901,0.987787,0.004504,0.007709,0.998949,0.001051,0.000000
3,0.000658,0.000086,0.999256,0.001490,0.000130,0.998380,0.000538,0.000138,0.999324,0.013634,0.004166,0.982199,0.000000,0.002127,0.997873
4,0.999840,0.000151,0.000009,0.999810,0.000178,0.000012,0.999880,0.000116,0.000004,0.988019,0.004456,0.007524,0.999069,0.000931,0.000000


# Machine Learning

In [6]:
def objective(trial, X, y):
    
    loss = trial.suggest_categorical("loss", ["log_loss", "modified_huber", "hinge", "perceptron"])
    penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])
    alpha = trial.suggest_float("alpha", 1e-9, 1e-1, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
    
    learning_rate = trial.suggest_categorical("learning_rate", ["optimal", "adaptive", "constant", "invscaling"])
    eta0 = trial.suggest_float("eta0", 1e-5, 1.0, log=True)
    power_t = trial.suggest_float("power_t", 0.1, 0.5) if learning_rate == "invscaling" else 0.5
    
    epsilon = trial.suggest_float("epsilon", 1e-3, 1e-1, log=True) 
    tol = trial.suggest_float("tol", 1e-6, 1e-2, log=True)
    average = trial.suggest_categorical("average", [True, False, 1, 5, 10]) # Permite médias baseadas em amostras

    class_weight = trial.suggest_categorical("class_weight", ["balanced", None])
    max_iter = trial.suggest_int("max_iter", 1000, 5000)

    w0 = trial.suggest_float('weight_class_0', 0.05, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.05, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.05, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train_fold, X_valid_fold = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]
        
        model = SGDClassifier(
            loss=loss,
            penalty=penalty,
            alpha=alpha,
            learning_rate=learning_rate,
            eta0=eta0,
            power_t=power_t,
            l1_ratio=l1_ratio,
            epsilon=epsilon,
            tol=tol,
            average=average,
            class_weight=class_weight,
            max_iter=max_iter,
            random_state=42,
            n_jobs=1
        ).fit(X_train_fold, y_train_fold)

        if loss in ["log_loss", "modified_huber"]:
            proba = model.predict_proba(X_valid_fold)
            weighted_probas = proba * weights
            pred = np.argmax(weighted_probas, axis=1)
        
        else:
            decision = model.decision_function(X_valid_fold)
            weighted_decision = decision * weights
            pred = np.argmax(weighted_decision, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.class_encoded), 
    n_trials=60, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-06-26 16:01:09,385] A new study created in memory with name: no-name-037b66cc-8f8a-4699-8f42-58136eb5b26f
Best trial: 11. Best value: 0.95523:   2%|██                                                                                                                            | 1/60 [00:13<13:19, 13.54s/it]

[I 2026-06-26 16:01:22,918] Trial 11 finished with value: 0.9552302339515817 and parameters: {'loss': 'perceptron', 'penalty': 'l2', 'alpha': 0.026338791137051495, 'l1_ratio': 0.23420408435467832, 'learning_rate': 'invscaling', 'eta0': 0.08662252522495897, 'power_t': 0.324085343787464, 'epsilon': 0.001878334942861441, 'tol': 0.000580389730867494, 'average': False, 'class_weight': 'balanced', 'max_iter': 3349, 'weight_class_0': 3.26451033075061, 'weight_class_1': 0.558984960183296, 'weight_class_2': 0.4151023653624717}. Best is trial 11 with value: 0.9552302339515817.


Best trial: 1. Best value: 0.958779:   3%|████▏                                                                                                                         | 2/60 [00:13<05:30,  5.70s/it]

[I 2026-06-26 16:01:23,135] Trial 1 finished with value: 0.9587791723826482 and parameters: {'loss': 'perceptron', 'penalty': 'l2', 'alpha': 5.900882007936968e-07, 'l1_ratio': 0.9148256430608815, 'learning_rate': 'constant', 'eta0': 0.015442668729119991, 'epsilon': 0.0021724364301837695, 'tol': 3.116989881656874e-05, 'average': 1, 'class_weight': None, 'max_iter': 1585, 'weight_class_0': 7.3304867100264675, 'weight_class_1': 3.013995623183563, 'weight_class_2': 0.18773380273754553}. Best is trial 1 with value: 0.9587791723826482.


Best trial: 1. Best value: 0.958779:   5%|██████▎                                                                                                                       | 3/60 [00:15<03:37,  3.82s/it]

...

[I 2026-06-26 16:12:05,096] Trial 54 finished with value: 0.9650363817898017 and parameters: {'loss': 'hinge', 'penalty': 'l1', 'alpha': 5.504406447375479e-09, 'l1_ratio': 0.6302792132230632, 'learning_rate': 'optimal', 'eta0': 6.369381400906912e-05, 'epsilon': 0.008164570562087588, 'tol': 0.001434970308348108, 'average': 1, 'class_weight': 'balanced', 'max_iter': 2905, 'weight_class_0': 0.13103548438569954, 'weight_class_1': 0.1297748122224722, 'weight_class_2': 1.3888363969434463}. Best is trial 43 with value: 0.9651826343119738.


Best trial: 43. Best value: 0.965183:  98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉  | 59/60 [11:03<00:14, 14.57s/it]

[I 2026-06-26 16:12:12,634] Trial 26 finished with value: 0.9649857983290072 and parameters: {'loss': 'hinge', 'penalty': 'elasticnet', 'alpha': 1.0164652601501406e-09, 'l1_ratio': 0.6217147014824294, 'learning_rate': 'optimal', 'eta0': 2.035870661892815e-05, 'epsilon': 0.01984965181360695, 'tol': 1.0349387693670746e-06, 'average': 1, 'class_weight': 'balanced', 'max_iter': 4048, 'weight_class_0': 0.4809187160421657, 'weight_class_1': 0.05304314600534901, 'weight_class_2': 0.9969268416943458}. Best is trial 43 with value: 0.9651826343119738.


Best trial: 43. Best value: 0.965183: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [12:00<00:00, 12.01s/it]

[I 2026-06-26 16:13:10,254] Trial 59 finished with value: 0.9651298059054303 and parameters: {'loss': 'hinge', 'penalty': 'l1', 'alpha': 4.3556894549052916e-09, 'l1_ratio': 0.7641990807157217, 'learning_rate': 'optimal', 'eta0': 0.0008201683031529088, 'epsilon': 0.020958397903830486, 'tol': 0.0009820596028975086, 'average': 1, 'class_weight': 'balanced', 'max_iter': 1390, 'weight_class_0': 0.13725246350847753, 'weight_class_1': 0.20226164009486253, 'weight_class_2': 0.23521733487234062}. Best is trial 43 with value: 0.9651826343119738.

Best trial score:
0.9651826343119738

Best params:
{'loss': 'hinge', 'penalty': 'l1', 'alpha': 9.689736169307372e-09, 'l1_ratio': 0.770062125020043, 'learning_rate': 'optimal', 'eta0': 0.00039626607675890006, 'epsilon': 0.015450027627626357, 'tol': 0.001122221127192132, 'average': True, 'class_weight': 'balanced', 'max_iter': 1876, 'weight_class_0': 0.09687765950579234, 'weight_class_1': 0.10015647111318432, 'weight_class_2': 1.5646633284138467}


In [7]:
# study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=500, n_jobs=-1, show_progress_bar=True)

# print("\nBest trial score:")
# print(study.best_trial.value)

# print("\nBest params:")
# print(study.best_trial.params)

In [8]:
sgd_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

sgd = SGDClassifier(
    **sgd_params,
    random_state=42,
    n_jobs=1
).fit(X_train, y_train.class_encoded)

loss = sgd_params['loss']

if loss in ["log_loss", "modified_huber"]:
    test_proba = sgd.predict_proba(X_test)

else:
    test_proba = sgd.decision_function(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [9]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [10]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_sgd.csv', index=False)

In [11]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [12]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0',
       'xgb_1', 'xgb_2', 'lg_0', 'lg_1', 'lg_2', 'sgd_0', 'sgd_1', 'sgd_2'],
      dtype='str')

In [13]:
len(study.trials)

60